# Notebook 8: Inference & Generation - JAX

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from jax import random

# Define a simple model for generation
class Generator(nn.Module):
    vocab_size: int
    hidden_size: int
    
    def setup(self):
        self.embedding = nn.Embed(self.vocab_size, self.hidden_size)
        self.dense = nn.Dense(self.hidden_size)
        self.lm_head = nn.Dense(self.vocab_size)
    
    def __call__(self, input_ids):
        x = self.embedding(input_ids)
        x = self.dense(x)
        return self.lm_head(x)

def greedy_decode(model, params, input_ids, max_new_tokens):
    """Greedy decoding in JAX"""
    generated = input_ids
    
    for _ in range(max_new_tokens):
        logits = model.apply(params, generated)
        next_token = jnp.argmax(logits[:, -1, :], axis=-1, keepdims=True)
        generated = jnp.concatenate([generated, next_token], axis=1)
    
    return generated

def beam_search(model, params, input_ids, beam_width=3, max_new_tokens=10):
    """Beam search in JAX"""
    sequences = [input_ids]
    scores = [0.0]
    
    for _ in range(max_new_tokens):
        candidates = []
        
        for seq, score in zip(sequences, scores):
            logits = model.apply(params, seq)
            next_logits = logits[:, -1, :]
            
            # Top-k
            topk_logits, topk_ids = jax.lax.top_k(next_logits, beam_width)
            log_probs = jnp.log(jax.nn.softmax(next_logits, axis=-1))
            topk_log_probs = jnp.take_along_axis(log_probs, topk_ids, axis=-1)
            
            for i in range(beam_width):
                new_seq = jnp.concatenate([seq, topk_ids[:, i:i+1]], axis=1)
                new_score = score + topk_log_probs[:, i]
                candidates.append((new_seq, new_score))
        
        # Keep top beams
        candidates.sort(key=lambda x: x[1][0], reverse=True)
        sequences = [c[0] for c in candidates[:beam_width]]
        scores = [c[1][0] for c in candidates[:beam_width]]
    
    return sequences[scores.index(max(scores))]

def temperature_sample(logits, temperature=1.0):
    """Temperature sampling"""
    if temperature == 0:
        return jnp.argmax(logits, axis=-1)
    
    probs = jax.nn.softmax(logits / temperature, axis=-1)
    return jax.random.categorical(jax.random.PRNGKey(0), logits / temperature, axis=-1)

# Test
model = Generator(vocab_size=1000, hidden_size=128)
rng = random.PRNGKey(42)
input_ids = jnp.array([[1, 2, 3, 4, 5]])

params = model.init(rng, input_ids)

# Greedy
greedy_out = greedy_decode(model, params, input_ids, 10)
print(f"Greedy: {greedy_out[0, :8].tolist()}")

# Temperature
logits = model.apply(params, input_ids)
temp_token = temperature_sample(logits[:, -1, :], temperature=0.8)
print(f"Temperature sample: {temp_token[0].tolist()}")

## Verification
1. ✅ Can implement greedy decoding in JAX
2. ✅ Can implement beam search in JAX
3. ✅ Can implement sampling methods